# Lekcija 4: Popunjavanje područja i povezane komponente

Kada dobijemo binarnu sliku (primjerice pragovanjem iz Lekcije 3), prirodno je postaviti sljedeće pitanje: *koliko zasebnih područja postoji i gdje se nalaze?* Na to odgovaraju dva postupka:

- **Popunjavanje područja** (engl. *flood fill*) širi područje iz jednog početnog piksela na sve povezane susjedne piksele slične vrijednosti.
- **Označavanje povezanih komponenti** istodobno pronalazi *sva* takva područja u binarnoj slici i svakom dodjeljuje jedinstvenu oznaku.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Izrada binarne slike s nekoliko zasebnih područja

Učitajmo sliku voća u sivim tonovima na tamnoj pozadini te primijenimo postupke iz Lekcije 3 kako bismo pragovali sliku i uklonili šum tipa sol, odnosno sitne bijele točkice.


In [ ]:
img = cv2.imread('../img/fruit.jpg', cv2.IMREAD_GRAYSCALE)
_, im_bin = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel = np.ones((3, 3), np.uint8)
im_bin2 = cv2.morphologyEx(im_bin, cv2.MORPH_OPEN, kernel)

fig, axes = plt.subplots(1, 3, figsize=(10,4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(im_bin, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Thresholded')
axes[1].axis('off')
axes[2].imshow(im_bin2, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('After opening')
axes[2].axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Izvor slike: Stan Birchfield</p>


## Popunjavanje područja iz početne točke

Popunjavanje područja započinje u početnom pikselu i širi se na sve povezane piksele čije su vrijednosti unutar dopuštenog odstupanja od vrijednosti početnog piksela, bojeći ih novom bojom. Klasični algoritam održava skup piksela koje još treba posjetiti, implementiran pomoću stoga ili reda. Uklonite piksel sa stoga; ako pripada objektu i još nije popunjen, označite ga kao popunjenog i dodajte na stog njegove susjede prema 4-povezanosti. Ponavljajte postupak dok stog ne postane prazan. Tada su pronađeni svi pikseli do kojih se iz početnog piksela može doći putem koji prolazi kroz piksele objekta.


In [ ]:
def flood_fill_stack(binary, seed):
    """Classic flood fill: an explicit stack holds the frontier of pixels still to visit."""
    filled = np.zeros_like(binary, dtype=bool)
    h, w = binary.shape
    stack = [seed]
    while stack:
        x, y = stack.pop()
        if x < 0 or x >= w or y < 0 or y >= h:
            continue
        if filled[y, x] or binary[y, x] == 0:
            continue
        filled[y, x] = True
        stack.extend([(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)])  # 4-connected neighbors
    return filled

Ovdje algoritam pokrećemo iz piksela unutar prve banane.


In [ ]:
seed = (60, 110)  # (x, y) inside the first banana
im_region_filled = flood_fill_stack(im_bin2, seed)

im_filled = cv2.cvtColor(im_bin2, cv2.COLOR_GRAY2RGB)
im_filled[im_region_filled] = (255, 140, 0)

plt.imshow(im_filled)
plt.scatter(*seed, c='red', s=30, marker='x')
plt.title(f'Flood fill from scratch ({im_region_filled.sum()} pixels filled)')
plt.axis('off')
plt.show()

## Popunjavanje područja iz početne točke

Funkcija `cv2.floodFill` radi potpuno isto, ali znatno brže jer je implementirana u prevedenom (kompajliranom) kodu.


In [ ]:
# floodFill needs a mask 2 pixels larger than the image, and modifies the image in place
im_filled_cv2 = cv2.cvtColor(im_bin2, cv2.COLOR_GRAY2RGB)
im_mask = np.zeros(np.add(im_bin2.shape, (2, 2)), dtype=np.uint8)

cv2.floodFill(im_filled_cv2, im_mask, seed, (255, 140, 0))

plt.imshow(im_filled_cv2)
plt.scatter(*seed, c='red', s=30, marker='x')
plt.title('Flood fill from one seed (red x)')
plt.axis('off')
plt.show()

im_region_filled_cv2 = np.all(im_filled_cv2 == (255, 140, 0), axis=-1)
print(f'The two implementations match exactly: {np.array_equal(im_region_filled, im_region_filled_cv2)}')

## Povezane komponente: označavanje svih područja odjednom

Umjesto ručnog odabira početnih piksela, algoritam za označavanje povezanih komponenti (`cv2.connectedComponentsWithStats`) pregledava cijelu sliku i svakom zasebnom području dodjeljuje vlastitu cjelobrojnu oznaku. Klasični algoritam, koji je radi sažetosti ovdje izostavljen, koristi strukturu *union-find* te zahtijeva dva prolaza kroz sliku i obilazak tablice ekvivalencija.


In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(im_bin2, connectivity=8)

# Give each label a distinct random color for visualization
colors = np.array([(0,0,0), (255,0,0), (0,200,0), (0,0,255), (180,180,0), (0,180,180), (180,0,180)])

im_colored = colors[labels].astype(np.uint8)

plt.imshow(im_colored)
for label in range(1, num_labels):
    cx, cy = centroids[label]
    plt.text(cx, cy, str(label), color='white', ha='center', va='center', fontsize=12, fontweight='bold')
plt.title('Connected components, colored and labeled')
plt.axis('off')
plt.show()

Algoritam za označavanje povezanih komponenti vraća i korisne statističke podatke, primjerice omeđujući pravokutnik, površinu i težište. Te dodatne informacije dobivamo gotovo bez dodatnog računanja, kao što ćemo vidjeti u Lekciji 5.


In [ ]:
print(f'Found {num_labels - 1} blobs (plus the background as label 0)\n')
print(f'{"label":>5} {"area":>6} {"centroid":>16}')
for label in range(1, num_labels):
    area = stats[label, cv2.CC_STAT_AREA]
    cx, cy = centroids[label]
    print(f'{label:>5} {area:>6} ({cx:6.1f}, {cy:6.1f})')

## Filtriranje područja prema veličini

Povezane komponente često se koriste za odbacivanje malih područja nalik šumu i zadržavanje samo značajnih područja. Ovdje odabiremo prag koji uklanja male jabuke (područja 2 i 6).


In [ ]:
min_area = 3000
img_minsize = np.zeros_like(im_bin2)
for label in range(1, num_labels):
    if stats[label, cv2.CC_STAT_AREA] > min_area:
        img_minsize[labels == label] = 255

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(im_bin2, cmap='gray')
axes[0].set_title('All blobs')
axes[0].axis('off')
axes[1].imshow(img_minsize, cmap='gray')
axes[1].set_title(f'Blobs with area > {min_area}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### Zadaci

1. Promijenite `connectivity=8` u `connectivity=4` u funkciji `connectedComponentsWithStats`. Izradite binarnu sliku (primjerice niz pojedinačnih piksela raspoređenih dijagonalno poput stepenica) za koju 4-povezanost i 8-povezanost daju *različit* broj komponenti.
2. Upotrijebite `cv2.floodFill` s tolerancijom `loDiff`/`upDiff` različitom od nule na slici u sivim tonovima (umjesto na binarnoj slici) i opišite što se mijenja.
